# First blood, role, rank, and victory

**Question:** Does first-blood role have an association with winning, and does the relationship differ across sampled rank groups?

This notebook reads a completed, provenance-tracked Riot collection. It never fabricates observations. See the project README for the operational current-anchor rank definition, patch confirmation, exclusions, and selection bias. Outputs are empty in source control until live data is collected.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from firstblood.storage import database, run_config
from firstblood.analysis import summarize, interaction_contrasts, make_charts

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from league-first-blood/ or notebooks/"
DB = ROOT / "data/firstblood.sqlite"
run_id = os.environ.get("FIRSTBLOOD_RUN_ID")
run = None
if DB.exists():
    with database(DB) as con:
        if not run_id:
            row = con.execute("SELECT run_id FROM runs WHERE status='complete' ORDER BY completed_at DESC LIMIT 1").fetchone()
            run_id = row[0] if row else None
        if run_id:
            run, config = run_config(con, run_id)
            if run['status'] != 'complete':
                raise ValueError("Selected collection is not complete")
if run is None:
    display(Markdown("**Pending live collection:** no completed Riot run. No empirical findings or charts are available."))
else:
    display(Markdown(f"**Run:** `{run_id}` · **Patch:** {run['selected_patch']} · **Platform:** {config['platform'].upper()}"))
    display(Markdown(f"**Collection UTC:** {run['started_at']} through {run['completed_at']}"))
    display(pd.Series(config, name="frozen_configuration").to_frame())
    display(pd.Series(json.loads(run['audit_json']), name="collection_audit").to_frame())

## SQL and match-level exploration

Each row is one deduplicated match. `run_matches.rank_group` is the run-specific anchor label; do not use an older run's label from `matches`.

Inspect match counts, roles, durations, and sampling-window coverage before comparing outcomes.

In [ ]:
if run is not None:
    with database(DB) as con:
        frame = pd.read_sql_query("""
            SELECT m.match_id, m.patch, rm.rank_group, m.role, m.first_blood_win,
                   m.duration_seconds, m.game_start_utc
            FROM matches m JOIN run_matches rm USING(match_id)
            WHERE rm.run_id=? AND m.patch=? ORDER BY m.match_id
        """, con, params=(run_id, run['selected_patch']))
        sql = (ROOT / "sql/analysis.sql").read_text().split(';')[0]
        sql_summary = pd.read_sql_query(sql, con, params={'run_id': run_id})
    assert not frame.empty and frame.match_id.is_unique
    assert frame.patch.nunique() == 1
    display(sql_summary)
    display(frame.groupby('rank_group').duration_seconds.describe())
    display(pd.crosstab(frame.rank_group, frame.role))
    display(pd.to_datetime(frame.game_start_utc, unit='s', utc=True).agg(['min', 'max']))

## Rates, 95% Wilson intervals, and role-by-rank patterns

Wilson intervals describe binomial uncertainty under independence; repeated players and convenience sampling can make them too narrow. The exploratory interaction contrasts are unadjusted approximations. CI overlap is not an interaction test, and the 50% line is not a causal baseline.

In [ ]:
if run is not None:
    summary = summarize(frame)
    assert summary.n.sum() == len(frame)
    display(summary)
    display(interaction_contrasts(summary))
    figures = ROOT / '.local/notebook-figures'
    make_charts(summary, figures, f"Patch {run['selected_patch']} · {len(frame):,} unique games")
    display(Image(filename=str(figures / 'win_rates.png')))
    display(Image(filename=str(figures / 'sample_sizes.png')))

## Interpretation and limitations

After a live run, compare effect sizes, interval widths, and sample composition. Check whether the role pattern changes by anchor-rank group without claiming historical lobby rank or causal effects.

Use `firstblood analyze --run-id <id>` to publish the observed aggregate report and exact collection metadata. With no completed run, the study remains blocked on live collection; this notebook makes no empirical conclusion.